# 03 — Build HMM-Ready Master Panel
**Healthy Purchase Cycle | NielsenIQ HMS | 2008–2017**

Build a memory-efficient, reproducible HMM input consisting of **two distinct layers**:

- **Trip emissions** (`hmm_trips/panel_year=YYYY/trips.parquet`): original six outcomes, category spending and observed trip-level food spending.
- **Annual transitions** (`hmm_annual_covariates.parquet`): household-year demographics, seven life events, and three controls.

The latent state is **household-year**; emissions occur for **each shopping trip**. Demographics are not repeated across millions of trips. `hhid` and `trip_id` stay strings; no 2007 trips are included. This notebook does **not** reconstruct events or reassign their timing.

**Important:** `food_spend` is sourced from Step 02, not derived from gated `y`. A trip missing from food-spending results remains `NULL`/unknown, not zero. Food-panel matching audits must be reviewed before estimation. Only nonpurchase outcomes have already been converted from category absence to `0` upstream.


In [ ]:
# Cell 1 — Packages, paths and safe DuckDB settings
from pathlib import Path
import time
import duckdb
import pandas as pd
from IPython.display import display

DATA = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
BASE = DATA / "processed" / "hmm_panel"
DEMOGRAPHICS = DATA / "processed" / "demographics" / "household_demographics_with_events_2008_2017.tsv"
CATEGORY_PANEL = BASE / "trip_purchase_panel_y0"
FOOD_PANEL = BASE / "trip_food_spending"
OUT = BASE / "hmm_master"
TRIP_OUT = OUT / "hmm_trips"
ANNUAL_OUT = OUT / "hmm_annual_covariates.parquet"
AUDIT = OUT / "validation"
TEMP = DATA / "temp" / "duckdb_hmm_master"
YEARS = range(2008, 2018)
MEMORY_LIMIT = "4GB"
THREADS = 4
# Require an explicit review of mismatched food records before model estimation.
MIN_FOOD_MATCH_RATE = 0.95  # Warning threshold only; never fills missing food amounts.

for path in (TRIP_OUT, AUDIT, TEMP):
    path.mkdir(parents=True, exist_ok=True)
assert DEMOGRAPHICS.is_file(), f"Missing Step 03 demographic output: {DEMOGRAPHICS}"
for year in YEARS:
    for label, folder in (("corrected purchases", CATEGORY_PANEL), ("food spending", FOOD_PANEL)):
        f = folder / f"panel_year={year}" / "trips.parquet"
        assert f.is_file(), f"Missing {label} for {year}: {f}"

def lit(path):
    return "'" + Path(path).as_posix().replace("'", "''") + "'"
def name(s):
    return '"' + s.replace('"', '""') + '"'
def pq(path):
    return f"read_parquet({lit(path)})"

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory={lit(TEMP)}")
print("DuckDB:", duckdb.__version__)
print("Annual source:", DEMOGRAPHICS)


## A — Validate annual demographics before building anything

Use the **seven** final life-event names (including `getting_divorce_or_separation`) from demographic Step 03. Keep the output compact: only model covariates and a transition-validity flag. The full, cleaned demographics remain in their original restricted TSV.


In [ ]:
# Cell 2 — Annual source schema and duplicate/key checks
EVENTS = [
    "event_new_child", "getting_married", "getting_divorce_or_separation",
    "moving", "getting_job", "retirement", "other_unemployment",
]
CONTROLS = ["K_1", "income10k", "family_size_numeric"]
EXTRA = ["valid_transition"]  # Included only if present in Step 03.
KEYS = ["hhid", "panel_year", "trip_id", "purchase_date"]
OUTCOMES = ["fv_y", "milk_y", "yogurt_y", "cereal_y", "snack_y", "beverage_y"]
SPENDS = [
    "fresh_spending", "non_fresh_spending",
    "skimmed_milk_spending", "low_fat_milk_spending", "other_milk_spending",
    "skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending",
    "low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending",
    "healthy_snack_spending", "unhealthy_snack_spending",
    "healthy_beverage_spending", "unhealthy_beverage_spending",
]
FOOD_FIELDS = ["regular_food_spend", "magnet_food_spend", "food_spend", "food_trip_matched"]

# IDs are explicitly read as VARCHAR; NULL is preserved for event/controls.
con.execute(f"""CREATE OR REPLACE VIEW demo_raw AS
    SELECT * FROM read_csv({lit(DEMOGRAPHICS)}, delim='\t', header=true, all_varchar=true)
""")
demo_cols = set(con.execute('DESCRIBE SELECT * FROM demo_raw').df().column_name)
required = {"hhid", "panel_year", *EVENTS, *CONTROLS}
assert required <= demo_cols, f"Missing demographic fields: {sorted(required - demo_cols)}"
optional = [x for x in EXTRA if x in demo_cols]
con.execute("""CREATE OR REPLACE VIEW demo_keys AS
    SELECT TRIM(hhid) AS hhid, TRY_CAST(panel_year AS INTEGER) AS panel_year
    FROM demo_raw""")
issues = con.execute("""SELECT
    COUNT(*) AS records,
    COUNT(*) FILTER (WHERE hhid IS NULL OR hhid='' OR panel_year IS NULL OR panel_year NOT BETWEEN 2008 AND 2017) AS bad_keys,
    COUNT(*) - COUNT(DISTINCT (hhid, panel_year)) AS duplicate_hh_years
    FROM demo_keys""").df()
display(issues)
assert issues.bad_keys.iloc[0] == 0, "Invalid demographic keys/years"
assert issues.duplicate_hh_years.iloc[0] == 0, "Nonunique demographic household-years"
# Numeric and event code validation. Never coerce malformed input to an event.
for field in EVENTS + ["K_1"]:
    bad = con.execute(f"""SELECT COUNT(*) FROM demo_raw
        WHERE {name(field)} IS NOT NULL AND TRIM({name(field)}) <> ''
          AND TRY_CAST({name(field)} AS INTEGER) NOT IN (0,1)
    """).fetchone()[0]
    assert bad == 0, f"Invalid {field} codes: {bad}"
print('Annual source validated.')


## B — Build annual covariates for households represented in the purchase trips

Create a distinct household-year index from all corrected purchase partitions and **join annual demographics one-to-one**. Retain a match indicator to audit excluded/no-demographic household-years. Only matched households enter the HMM outputs. Never interpret absent life-event information as zero.


In [ ]:
# Cell 3 — Small household-year index and model-ready annual covariates
trip_paths = [CATEGORY_PANEL / f"panel_year={year}" / "trips.parquet" for year in YEARS]
trip_rel = 'read_parquet([' + ','.join(lit(x) for x in trip_paths) + '])'
con.execute(f"""CREATE OR REPLACE TEMP TABLE observed_hh_years AS
    SELECT DISTINCT CAST(hhid AS VARCHAR) AS hhid, CAST(panel_year AS INTEGER) AS panel_year
    FROM {trip_rel}
""")
# Check annual coverage independently of trip-amount matching.
coverage = con.execute("""SELECT o.panel_year,
    COUNT(*) AS purchase_household_years,
    COUNT(*) FILTER (WHERE d.hhid IS NOT NULL) AS matched_demographic_years,
    COUNT(*) FILTER (WHERE d.hhid IS NULL) AS unmatched_demographic_years
    FROM observed_hh_years o
    LEFT JOIN demo_keys d USING (hhid, panel_year)
    GROUP BY o.panel_year ORDER BY o.panel_year""").df()
display(coverage)
coverage.to_csv(AUDIT / 'annual_demographic_coverage.csv', index=False)

# Use the strict matched intersection. This excludes households removed in Step 02
# and household-years for which annual demographic extraction is unavailable.
# Cast the seven events to nullable integer, controls to numeric.
fields = ["k.hhid", "k.panel_year"]
fields += [f"TRY_CAST(NULLIF(TRIM(d.{name(x)}), '') AS INTEGER) AS {name(x)}" for x in EVENTS]
fields += ["TRY_CAST(NULLIF(TRIM(d.K_1), '') AS INTEGER) AS K_1",
           "TRY_CAST(NULLIF(TRIM(d.income10k), '') AS DOUBLE) AS income10k",
           "TRY_CAST(NULLIF(TRIM(d.family_size_numeric), '') AS DOUBLE) AS family_size_numeric"]
if "valid_transition" in optional:
    fields.append("TRY_CAST(NULLIF(TRIM(d.valid_transition), '') AS BOOLEAN) AS valid_transition")

con.execute(f"""COPY (
    SELECT {', '.join(fields)}
    FROM observed_hh_years k INNER JOIN demo_raw d
      ON k.hhid=TRIM(d.hhid) AND k.panel_year=TRY_CAST(d.panel_year AS INTEGER)
) TO {lit(ANNUAL_OUT)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
annual = pq(ANNUAL_OUT)
annual_check = con.execute(f"""SELECT COUNT(*) household_years, COUNT(DISTINCT hhid) households,
    COUNT(*)-COUNT(DISTINCT (hhid,panel_year)) duplicate_keys
    FROM {annual}""").df()
display(annual_check)
assert annual_check.duplicate_keys.iloc[0] == 0
print('Annual covariates:', ANNUAL_OUT)


## C — Join annual eligibility with trip emissions and trip food spending

Trip source: six-category corrected Parquet (`0` indicates nonpurchase). Food source: Step 02 trip-level amounts. Each is independently required to have **unique four-column trip keys**. The join is left-sided from the emission universe and filtered against the matched annual index; food spending remains missing if the Step 02 row is missing or did not match the original purchases.

We do **not** replace missing `food_spend` with 0, and do **not** construct `healthy_food_spend` by applying `category_y` to category spending.


In [ ]:
# Cell 4 — Annual partition loop: validation, join and audit
EXPECTED = {'cereal_y': (0,1,2,3), **{y:(0,1,2) for y in OUTCOMES if y!='cereal_y'}}
rows = []
for year in YEARS:
    started = time.perf_counter()
    cfile = CATEGORY_PANEL / f'panel_year={year}' / 'trips.parquet'
    ffile = FOOD_PANEL / f'panel_year={year}' / 'trips.parquet'
    c, f = pq(cfile), pq(ffile)
    ccols = set(con.execute(f'DESCRIBE SELECT * FROM {c}').df().column_name)
    fcols = set(con.execute(f'DESCRIBE SELECT * FROM {f}').df().column_name)
    assert set(KEYS + OUTCOMES + SPENDS) <= ccols, (year, 'category fields', sorted(set(KEYS+OUTCOMES+SPENDS)-ccols))
    assert set(KEYS + FOOD_FIELDS) <= fcols, (year, 'food fields', sorted(set(KEYS+FOOD_FIELDS)-fcols))

    # Trip identifiers remain strings; DATE cast only for date comparison.
    def view_sql(rel):
        return f"""SELECT CAST(hhid AS VARCHAR) hhid, CAST(panel_year AS INTEGER) panel_year,
                  CAST(trip_id AS VARCHAR) trip_id, TRY_CAST(purchase_date AS DATE) purchase_date,
                  * EXCLUDE (hhid, panel_year, trip_id, purchase_date)
                  FROM {rel}"""
    con.execute(f'CREATE OR REPLACE TEMP VIEW cat_y AS {view_sql(c)}')
    con.execute(f'CREATE OR REPLACE TEMP VIEW food_y AS {view_sql(f)}')
    checks = []
    for view in ('cat_y','food_y'):
        count, bad, duplicate = con.execute(f"""SELECT COUNT(*),
          COUNT(*) FILTER (WHERE hhid IS NULL OR TRIM(hhid)='' OR trip_id IS NULL OR TRIM(trip_id)=''
              OR purchase_date IS NULL OR panel_year <> {year} OR YEAR(purchase_date) <> {year}),
          COUNT(*)-COUNT(DISTINCT (hhid,panel_year,trip_id,purchase_date))
          FROM {view}""").fetchone()
        assert bad == 0 and duplicate == 0, (year,view,'invalid key',bad,'duplicate',duplicate)
        checks.append(count)
    bad_y = con.execute('SELECT ' + ','.join(
        f'COUNT(*) FILTER (WHERE {name(y)} IS NULL OR {name(y)} NOT IN ({",".join(map(str,EXPECTED[y]))}))'
        for y in OUTCOMES) + ' FROM cat_y').fetchone()
    assert not any(bad_y), (year, dict(zip(OUTCOMES,bad_y)))
    # The food table must not multiply category trips; missing food is preserved.
    target = TRIP_OUT / f'panel_year={year}' / 'trips.parquet'
    target.parent.mkdir(parents=True, exist_ok=True)
    selection = ', '.join('c.' + name(x) for x in OUTCOMES + SPENDS)
    con.execute(f"""COPY (
      SELECT c.hhid, c.panel_year, c.trip_id, c.purchase_date,
             {selection},
             f.regular_food_spend, f.magnet_food_spend, f.food_spend,
             f.food_trip_matched
      FROM cat_y c
      INNER JOIN {annual} a ON c.hhid=a.hhid AND c.panel_year=a.panel_year
      LEFT JOIN food_y f ON c.hhid=f.hhid AND c.panel_year=f.panel_year
        AND c.trip_id=f.trip_id AND c.purchase_date=f.purchase_date
    ) TO {lit(target)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    o = pq(target)
    metrics = con.execute(f"""SELECT COUNT(*) AS output_trips,
      COUNT(*)-COUNT(DISTINCT (hhid,panel_year,trip_id,purchase_date)) AS duplicate_output_keys,
      COUNT(*) FILTER (WHERE food_trip_matched IS TRUE) AS food_source_matched,
      COUNT(*) FILTER (WHERE food_trip_matched IS DISTINCT FROM TRUE) AS food_source_unmatched,
      COUNT(*) FILTER (WHERE food_spend IS NULL) AS null_food_spend,
      COUNT(*) FILTER (WHERE food_spend < 0 OR regular_food_spend < 0 OR magnet_food_spend < 0) AS negative_food_spend,
      COUNT(*) FILTER (WHERE food_trip_matched IS TRUE AND
          ABS(food_spend - (regular_food_spend + magnet_food_spend)) > 0.0001) AS inconsistent_food_totals
      FROM {o}""").fetchone()
    keys = ['output_trips','duplicate_output_keys','food_source_matched','food_source_unmatched',
            'null_food_spend','negative_food_spend','inconsistent_food_totals']
    r = dict(zip(keys,metrics))
    assert r['duplicate_output_keys']==0 and r['negative_food_spend']==0 and r['inconsistent_food_totals']==0, (year,r)
    r.update(panel_year=year,category_input_trips=checks[0],food_input_trips=checks[1],
             excluded_for_no_annual=checks[0]-r['output_trips'])
    r['food_match_rate'] = r['food_source_matched']/r['output_trips'] if r['output_trips'] else None
    r['seconds'] = round(time.perf_counter()-started,1)
    rows.append(r)
    print(f'{year}: {r["output_trips"]:,} trips, food match {r["food_match_rate"]:.1%}')
    if r['food_match_rate'] < MIN_FOOD_MATCH_RATE:
        print(f'  WARNING: food matching below {MIN_FOOD_MATCH_RATE:.0%}. Review Step 02 audit.')
    pd.DataFrame(rows).to_csv(AUDIT/'hmm_master_trip_merge_audit.csv',index=False)
display(pd.DataFrame(rows))


## D — Final cross-layer checks and estimation readiness

This checks that every output trip has exactly one annual household-year, and that no annual household-year was accidentally duplicated by the trip join. Missing event covariates are **not** zero-imputed. Check `moving` missingness before passing covariates to a model whose transition design requires complete cases.


In [ ]:
# Cell 5 — Final checks and concise model-preparation audit
hmm_paths = [TRIP_OUT/f'panel_year={year}'/'trips.parquet' for year in YEARS]
hmm_rel = 'read_parquet([' + ','.join(lit(p) for p in hmm_paths) + '])'
summary = con.execute(f"""WITH
    t AS (SELECT COUNT(*) trips, COUNT(DISTINCT hhid) households,
          COUNT(DISTINCT (hhid,panel_year)) trip_household_years
          FROM {hmm_rel}),
    a AS (SELECT COUNT(*) annual_rows, COUNT(DISTINCT hhid) annual_households FROM {annual})
    SELECT * FROM t CROSS JOIN a""").df()
display(summary)
assert summary.trip_household_years.iloc[0] == summary.annual_rows.iloc[0], 'Annual/trip household-year mismatch'
missing = con.execute('SELECT ' + ','.join(
    f'COUNT(*) FILTER (WHERE {name(x)} IS NULL) AS {name(x+"_missing")}'
    for x in EVENTS+CONTROLS) + f' FROM {annual}').df()
display(missing.T.rename(columns={0:'missing_household_years'}))
summary.to_csv(AUDIT/'hmm_master_overall_summary.csv', index=False)
missing.to_csv(AUDIT/'hmm_master_annual_missingness.csv', index=False)
print('READY FOR REVIEW — trip/annual indexing validated. Model estimation should use')
print('  Annual :', ANNUAL_OUT)
print('  Trips  :', TRIP_OUT)
print('Review the food match rates and missing annual transition covariates before fitting the HMM.')
